%md
## Ingest constructors.json file

1. Read the file using spark dataframe reader API
2. Add Metadata Columns
- Source File
- Ingestion Timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/constructors.json"

target_path = "s3://bizf1project123/Bronze/Kalashri_increment/constructors"

## Step 1 - Read the CSV file using the dataframe reader API

In [0]:
# Define the schema
constructors_schema = """constructorId STRING, 
                         name STRING, 
                         nationality STRING, 
                         url STRING
                         """


In [0]:
# Read data from the constructors file
constructors_df = (
    spark.read
       .format('json')
       .schema(constructors_schema)
       .option('mode', 'FAILFAST')
       .load(source_file)
)

In [0]:
display(constructors_df)

%md
## Step 2 - Add Metadata Columns


- Source File

- Ingestion Timestamp

In [0]:
constructors_final_df = add_ingestion_metadata(constructors_df)

In [0]:
display(constructors_final_df)

## Step 3 - Write to bronze delta table

In [0]:
write_to_bronze(
    input_df=constructors_final_df,
    target_path=target_path,
    batch_id=v_batch_id
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Bronze/Kalashri_increment/constructors`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Bronze/Kalashri_increment/constructors",
#     True
# )